In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timezone
import time

sns.set_theme(style="whitegrid")
DATA_DIR = Path("dados_pokemon")
DATA_DIR.mkdir(exist_ok=True)
ATRIBUTOS = {"hp": "hp", "attack": "ataque", "defense": "defesa",
             "special-attack": "ataque_especial", "special-defense": "defesa_especial",
             "speed": "velocidade"}
COLS = list(ATRIBUTOS.values())


In [ ]:
# Coleta dos 151 registros
sessao = requests.Session()
sessao.headers.update({"User-Agent": "ProjetoAcademicoPokemon/1.0"})
registros, falhas = [], []
inicio_coleta = datetime.now(timezone.utc)
for pokemon_id in range(1, 152):
    try:
        resposta = None
        for tentativa in range(3):
            try:
                resposta = sessao.get(f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}", timeout=20)
                resposta.raise_for_status()
                break
            except requests.RequestException:
                if tentativa == 2:
                    raise
                time.sleep(1 + tentativa)
        item = resposta.json()
        tipos = [t["type"]["name"] for t in sorted(item["types"], key=lambda x: x["slot"])]
        stats = {s["stat"]["name"]: s["base_stat"] for s in item["stats"]}
        linha = {"id": item["id"], "nome": item["name"],
                 "tipo_primario": tipos[0],
                 "tipo_secundario": tipos[1] if len(tipos) > 1 else None}
        linha.update({col: stats.get(chave) for chave, col in ATRIBUTOS.items()})
        registros.append(linha)
    except (requests.RequestException, KeyError, IndexError, ValueError) as erro:
        falhas.append({"id": pokemon_id, "erro": str(erro)})
fim_coleta = datetime.now(timezone.utc)
df = pd.DataFrame(registros)
print(f"Coleta finalizada! {len(df)} de 151 Pokémon coletados com sucesso. Falhas: {len(falhas)}")
if falhas:
    print(pd.DataFrame(falhas))
if df.empty:
    raise RuntimeError("Nenhum registro foi coletado. Verifique a conexão e tente novamente.")
print(df.head())


In [ ]:
# Validação de integridade e limpeza
df = df.drop_duplicates(subset="id").sort_values("id").reset_index(drop=True)
for col in COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("IDs ausentes:", sorted(set(range(1, 152)) - set(df["id"])))
print("Duplicatas:", df["id"].duplicated().sum())
print("Valores ausentes por coluna:")
print(df.isna().sum().to_frame("ausentes"))
# O tipo secundário ausente é esperado em Pokémon de tipo único.
if df[COLS + ["tipo_primario"]].isna().any().any():
    raise ValueError("Há atributos ou tipos primários ausentes. Revise a coleta.")
if len(df) != 151:
    print("ATENÇÃO: coleta incompleta; resultados representam apenas os registros disponíveis.")

tipos_longos = df.melt(id_vars=["id", "nome"],
                       value_vars=["tipo_primario", "tipo_secundario"],
                       value_name="tipo").dropna(subset=["tipo"])[["id", "nome", "tipo"]]
print("Total de associações Pokémon-tipo:", len(tipos_longos))

arquivo = DATA_DIR / "pokemon_geracao_1.csv"
df.to_csv(arquivo, index=False, encoding="utf-8-sig")
print("Data UTC da coleta:", inicio_coleta.isoformat(), "até", fim_coleta.isoformat())
print("Total de registros:", len(df))
print("Total de colunas:", df.shape[1])
print("Tamanho do arquivo (KB):", round(arquivo.stat().st_size / 1024, 2))
print("Arquivo:", arquivo.resolve())
print(df.head(10))

estatisticas = df[COLS].describe().T
estatisticas["variancia"] = df[COLS].var()
print(estatisticas.round(2))
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df[COLS], ax=ax, color="#8fc8ee")
ax.set_title("Distribuição dos seis atributos — primeira geração")
ax.set_ylabel("Valor base do atributo")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

frequencia_tipos = tipos_longos["tipo"].value_counts()
print(frequencia_tipos.rename_axis("tipo").reset_index(name="quantidade"))
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=frequencia_tipos.values, y=frequencia_tipos.index, ax=ax, color="#76b5d8")
ax.set(title="Distribuição por tipo — contando tipos primário e secundário",
       xlabel="Número de Pokémon que possuem o tipo", ylabel="Tipo")
plt.tight_layout()
plt.show()

contagem_primaria = df.groupby("tipo_primario").size().rename("n_pokemon")
medias_por_tipo = df.groupby("tipo_primario")[COLS].mean().round(2)
resumo_por_tipo = medias_por_tipo.join(contagem_primaria).sort_values("n_pokemon", ascending=False)
print(resumo_por_tipo)
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(medias_por_tipo, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set(title="Média dos atributos por tipo primário", xlabel="Atributo", ylabel="Tipo primário")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df, x="tipo_primario", y="ataque", color="#a3d5ad", ax=ax)
ax.set(title="Distribuição do ataque por tipo primário", xlabel="Tipo primário", ylabel="Ataque base")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

matriz_correlacao = df[COLS].corr(method="pearson")
print(matriz_correlacao.round(3))
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(matriz_correlacao, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title("Correlação de Pearson entre atributos")
plt.tight_layout()
plt.show()
pares = (matriz_correlacao.where(np.triu(np.ones(matriz_correlacao.shape), k=1).astype(bool))
         .stack().rename("correlacao").reset_index())
pares.columns = ["atributo_1", "atributo_2", "correlacao"]
pares["correlacao_absoluta"] = pares["correlacao"].abs()
print(pares.sort_values("correlacao_absoluta", ascending=False).head(5))

mais_frequente = frequencia_tipos.idxmax()
mais_raro = frequencia_tipos.idxmin()
maior_media_ataque = medias_por_tipo["ataque"].idxmax()
maior_par = pares.sort_values("correlacao_absoluta", ascending=False).iloc[0]
print(f"1. O tipo mais frequente é {mais_frequente} ({frequencia_tipos.max()} Pokémon); "
      f"o menos frequente é {mais_raro} ({frequencia_tipos.min()} Pokémon).")
print(f"2. O tipo primário com maior média de ataque é {maior_media_ataque} "
      f"({medias_por_tipo.loc[maior_media_ataque, 'ataque']:.2f}), "
      f"com {contagem_primaria.loc[maior_media_ataque]} Pokémon no grupo.")
print(f"3. A maior correlação absoluta é entre {maior_par['atributo_1']} e "
      f"{maior_par['atributo_2']}: r = {maior_par['correlacao']:.3f}.")
